In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

In [ ]:
# Connect to OpenAI and Anthropic using the same OpenAI-compatible client trick from the lesson

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)

GPT_MODEL = "gpt-5"
CLAUDE_MODEL = "claude-sonnet-4-5-20250929"

system_message = "You are a helpful assistant that responds in markdown without code blocks"

In [ ]:
# Two streaming generators, straight from the Day 2 lesson

def stream_gpt(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(
        model=GPT_MODEL,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


def stream_claude(prompt):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = anthropic.chat.completions.create(
        model=CLAUDE_MODEL,
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
# Pull from both generators in lockstep, yielding the latest values from each.
# This is what actually lets both Markdown panes update concurrently.

def interleave_streams(gpt_gen, claude_gen):
    gpt_result, claude_result = "", ""
    gpt_done = claude_done = False
    while not (gpt_done and claude_done):
        if not gpt_done:
            try:
                gpt_result = next(gpt_gen)
            except StopIteration:
                gpt_done = True
        if not claude_done:
            try:
                claude_result = next(claude_gen)
            except StopIteration:
                claude_done = True
        yield gpt_result, claude_result

In [ ]:
vote_tally = {"GPT": 0, "Claude": 0}


def tally_markdown():
    return f"**Votes so far** — GPT: {vote_tally['GPT']} | Claude: {vote_tally['Claude']}"


def run_arena(prompt):
    yield "", "", tally_markdown()
    for gpt_result, claude_result in interleave_streams(stream_gpt(prompt), stream_claude(prompt)):
        yield gpt_result, claude_result, tally_markdown()


def vote(choice):
    vote_tally[choice] += 1
    return tally_markdown()


with gr.Blocks(title="Model Arena: GPT vs Claude") as arena:
    gr.Markdown("# Model Arena: GPT vs Claude \U0001f94a")
    prompt_box = gr.Textbox(label="Your prompt:", lines=4)
    run_button = gr.Button("Ask both models")
    with gr.Row():
        gpt_pane = gr.Markdown(label=GPT_MODEL)
        claude_pane = gr.Markdown(label=CLAUDE_MODEL)
    with gr.Row():
        gpt_vote_button = gr.Button("\U0001f44d GPT was better")
        claude_vote_button = gr.Button("\U0001f44d Claude was better")
    tally_display = gr.Markdown(tally_markdown())

    run_button.click(run_arena, inputs=prompt_box, outputs=[gpt_pane, claude_pane, tally_display])
    gpt_vote_button.click(lambda: vote("GPT"), outputs=tally_display)
    claude_vote_button.click(lambda: vote("Claude"), outputs=tally_display)

arena.launch()